# Premium Customer Intelligence: EDA, Modeling, and Explainability

This notebook builds a full insurance analytics workflow to answer:

**Which customers are most valuable and how can the business identify, retain, and grow them?**

Outputs align with director-ready deliverables:
- Portfolio EDA
- Customer-level CLV aggregation
- Premium customer classification
- SHAP explainability
- Business actions for retention, growth, and risk management


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_preparation import load_policy_data, aggregate_customer_level

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = PROJECT_ROOT / 'data' / 'clv_realistic_50000_5yr.csv'
policy_df = load_policy_data(DATA_PATH)
policy_df.head()


## 1. Dataset Overview

In [ ]:
print(f'Shape: {policy_df.shape}')
print(f'Unique customers: {policy_df.CustomerID.nunique():,}')
print(f'Years covered: {policy_df.Year.min()} - {policy_df.Year.max()}')

overview = pd.DataFrame({
    'column': policy_df.columns,
    'dtype': policy_df.dtypes.astype(str).values,
    'missing_pct': (policy_df.isna().mean() * 100).round(2).values,
    'unique_values': [policy_df[c].nunique(dropna=True) for c in policy_df.columns]
}).sort_values('missing_pct', ascending=False)
overview.head(20)


In [ ]:
policy_df.describe(include='all').T.head(25)

## 2. Portfolio Overview

In [ ]:
portfolio_metrics = {
    'Avg Written Premium': policy_df['DIRECTWRITTENPREMIUM_AM'].mean(),
    'Avg Earned Premium': policy_df['EARNEDPREMIUM_AM'].mean(),
    'Avg Net Loss': policy_df['NETLOSS_PAID_AM'].mean(),
    'Avg Claim Count': policy_df['CLAIMCOUNT_CT'].mean(),
    'Renewal Rate': policy_df['POLICY_RENEWED_FLAG'].mean(),
    'Delinquency Rate': policy_df['DelequencyFlag'].mean(),
    'Avg Satisfaction': policy_df['CustomerSatisfaction'].mean(),
}
pd.Series(portfolio_metrics).to_frame('value')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))

sns.histplot(policy_df['DIRECTWRITTENPREMIUM_AM'], bins=60, kde=True, ax=axes[0,0], color='#0a3d62')
axes[0,0].set_title('Written Premium Distribution')

sns.histplot(policy_df['EARNEDPREMIUM_AM'], bins=60, kde=True, ax=axes[0,1], color='#2e86de')
axes[0,1].set_title('Earned Premium Distribution')

sns.histplot(policy_df['NETLOSS_PAID_AM'], bins=60, kde=True, ax=axes[0,2], color='#e17055')
axes[0,2].set_title('Net Loss Distribution')

sns.histplot(policy_df['CLAIMCOUNT_CT'], bins=20, ax=axes[1,0], color='#16a085')
axes[1,0].set_title('Claim Count Distribution')

sns.histplot(policy_df['CustomerSatisfaction'], bins=10, ax=axes[1,1], color='#6c5ce7')
axes[1,1].set_title('Customer Satisfaction Distribution')

renewal_delq = pd.DataFrame({
    'metric': ['Renewal Rate', 'Delinquency Rate'],
    'value': [policy_df['POLICY_RENEWED_FLAG'].mean(), policy_df['DelequencyFlag'].mean()]
})
sns.barplot(data=renewal_delq, x='metric', y='value', ax=axes[1,2], palette=['#0a3d62', '#e17055'])
axes[1,2].set_ylim(0,1)
axes[1,2].set_title('Renewal vs Delinquency Rate')

plt.tight_layout()


## 3. Business Relationship Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))

sns.scatterplot(data=policy_df.sample(min(12000, len(policy_df)), random_state=42), x='PropertyValue', y='CoverageAmount', alpha=0.25, ax=axes[0,0], color='#0a3d62')
axes[0,0].set_title('Property Value vs Coverage Amount')

sns.scatterplot(data=policy_df.sample(min(12000, len(policy_df)), random_state=42), x='CoverageAmount', y='DIRECTWRITTENPREMIUM_AM', alpha=0.25, ax=axes[0,1], color='#2e86de')
axes[0,1].set_title('Coverage Amount vs Premium')

sns.regplot(data=policy_df.sample(min(12000, len(policy_df)), random_state=42), x='HAZARD_SCORE', y='CLAIMCOUNT_CT', scatter_kws={'alpha':0.2}, line_kws={'color':'#e17055'}, ax=axes[0,2])
axes[0,2].set_title('Hazard Score vs Claim Count')

sns.regplot(data=policy_df.sample(min(12000, len(policy_df)), random_state=42), x='ComplaintCount', y='CustomerSatisfaction', scatter_kws={'alpha':0.2}, line_kws={'color':'#d63031'}, ax=axes[1,0])
axes[1,0].set_title('Complaint Count vs Satisfaction')

sns.boxplot(data=policy_df, x='DelequencyFlag', y='PaymentDelayDays', ax=axes[1,1], palette=['#74b9ff', '#e17055'])
axes[1,1].set_title('Payment Delay by Delinquency Flag')

sns.boxplot(data=policy_df, x='POLICY_RENEWED_FLAG', y='CustomerSatisfaction', ax=axes[1,2], palette=['#e17055', '#00b894'])
axes[1,2].set_title('Satisfaction by Renewal Outcome')

plt.tight_layout()


In [ ]:
corr_checks = {
    'PropertyValue vs CoverageAmount': policy_df['PropertyValue'].corr(policy_df['CoverageAmount']),
    'CoverageAmount vs Premium': policy_df['CoverageAmount'].corr(policy_df['DIRECTWRITTENPREMIUM_AM']),
    'HazardScore vs ClaimCount': policy_df['HAZARD_SCORE'].corr(policy_df['CLAIMCOUNT_CT']),
    'ComplaintCount vs CustomerSatisfaction': policy_df['ComplaintCount'].corr(policy_df['CustomerSatisfaction']),
    'PaymentDelayDays vs DelequencyFlag': policy_df['PaymentDelayDays'].corr(policy_df['DelequencyFlag']),
    'CustomerSatisfaction vs POLICY_RENEWED_FLAG': policy_df['CustomerSatisfaction'].corr(policy_df['POLICY_RENEWED_FLAG'])
}
pd.Series(corr_checks).sort_values(ascending=False)


## 4. Time Analysis

In [ ]:
yearly = policy_df.groupby('Year', as_index=False).agg(
    yearly_premium=('EARNEDPREMIUM_AM', 'sum'),
    yearly_loss=('NETLOSS_PAID_AM', 'sum'),
    yearly_claims=('CLAIMCOUNT_CT', 'sum'),
    yearly_renewal=('POLICY_RENEWED_FLAG', 'mean')
)
yearly['yearly_margin'] = yearly['yearly_premium'] - yearly['yearly_loss']
yearly


In [ ]:
fig = px.line(
    yearly,
    x='Year',
    y=['yearly_premium', 'yearly_loss', 'yearly_margin'],
    markers=True,
    title='Yearly Premium, Loss, and Margin Trend',
)
fig.show()

fig_claims = px.bar(yearly, x='Year', y='yearly_claims', title='Yearly Claims Trend')
fig_claims.show()

fig_renewal = px.line(yearly, x='Year', y='yearly_renewal', markers=True, title='Yearly Renewal Trend')
fig_renewal.update_yaxes(tickformat='.0%')
fig_renewal.show()


## 5. Segment Analysis

In [ ]:
segment_views = {
    'State': 'POLICYRATEDSTATE_TP',
    'Income Bracket': 'IncomeBracket',
    'Payment Method': 'PaymentMethod',
    'Marketing Channel': 'MarketingChannel',
    'Product Subtype': 'PROPERTYCOVERAGESUBTYPE_TP'
}

segment_frames = []
for label, column in segment_views.items():
    seg = (policy_df.groupby(column, as_index=False)
        .agg(
            earned_premium=('EARNEDPREMIUM_AM', 'sum'),
            net_loss=('NETLOSS_PAID_AM', 'sum'),
            renewal_rate=('POLICY_RENEWED_FLAG', 'mean'),
            delinquency_rate=('DelequencyFlag', 'mean'),
            avg_satisfaction=('CustomerSatisfaction', 'mean'),
            records=('CustomerID', 'size')
        )
    )
    seg['segment_type'] = label
    seg = seg.rename(columns={column: 'segment_value'})
    segment_frames.append(seg)

segment_df = pd.concat(segment_frames, ignore_index=True)
segment_df.head(10)


In [ ]:
for seg_type in ['State', 'Income Bracket', 'Payment Method', 'Marketing Channel', 'Product Subtype']:
    view = segment_df[segment_df['segment_type'] == seg_type].sort_values('earned_premium', ascending=False).head(12)
    fig = px.bar(view, x='segment_value', y='earned_premium', color='renewal_rate',
                 color_continuous_scale='Blues', title=f'{seg_type}: Earned Premium and Renewal Strength')
    fig.update_layout(xaxis_title='', yaxis_title='Earned Premium')
    fig.show()


## 6. Customer-Level Aggregation and CLV Target

In [ ]:
customer_df = aggregate_customer_level(policy_df, premium_quantile=0.80)
customer_df.head()

In [ ]:
print(f'Customer-level rows: {len(customer_df):,}')
print(f'Premium share: {customer_df.premium_customer.mean():.2%}')
print('Value segment mix:')
print(customer_df.value_segment.value_counts(normalize=True).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(customer_df['customer_clv'], bins=60, kde=True, ax=axes[0], color='#0a3d62')
axes[0].set_title('Customer CLV Distribution')

segment_mix = customer_df['value_segment'].value_counts().reset_index()
segment_mix.columns = ['segment', 'customers']
sns.barplot(data=segment_mix, x='segment', y='customers', ax=axes[1], palette=['#0a3d62', '#74b9ff', '#e17055'])
axes[1].set_title('Customer Value Segments')

plt.tight_layout()


## 7. Modeling Pipeline (via Scripts)

In [ ]:
# Optional: run end-to-end scripts from notebook
# !python ../src/data_preparation.py --premium-quantile 0.80
# !python ../src/train_model.py
# !python ../src/explain_model.py

comparison_path = PROJECT_ROOT / 'artifacts' / 'model_comparison.csv'
metrics_path = PROJECT_ROOT / 'artifacts' / 'model_metrics.json'
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    display(comparison_df)
else:
    print('Run train_model.py to generate model comparison artifacts.')


## 8. SHAP Explainability Review

In [ ]:
shap_importance_path = PROJECT_ROOT / 'artifacts' / 'shap_feature_importance.csv'
if shap_importance_path.exists():
    shap_importance_df = pd.read_csv(shap_importance_path)
    display(shap_importance_df.head(15))
else:
    print('Run explain_model.py to generate SHAP outputs.')


In [ ]:
from IPython.display import Image, display
shap_image_path = PROJECT_ROOT / 'artifacts' / 'shap_summary.png'
if shap_image_path.exists():
    display(Image(filename=str(shap_image_path)))
else:
    print('No SHAP summary image found.')


## 9. Business Insights for Leadership

1. Portfolio economics are imbalanced: a minority of customers contribute disproportionate long-term value.
2. CLV-based segmentation provides a practical way to prioritize retention and growth investments.
3. The predictive model makes premium-customer identification operational, not anecdotal.
4. Explainability clarifies which levers increase premium-customer likelihood (retention, satisfaction, value profile) and which destroy value (claims burden, payment friction, delinquency risk).
5. Action bands (`Retain & Grow`, `Protect`, `Monitor / Re-price`) convert analytics into scalable decision workflows.
